# Notebook 00: Setup and Connection Verification
## Purpose: Verify connections from Jupyter to PostgreSQL and MinIO

**Connections:**
- PostgreSQL: `postgres` host (Docker network), user `analyst`, password `analyst_pass`, DB `oilfield`
- MinIO S3 API: `minio:9000`, access key `minioadmin`, secret key `minioadmin`

In [4]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text
import boto3
from io import BytesIO
import pyarrow as pa
import pyarrow.parquet as pq

print("All imports successful")

All imports successful


## 1. PostgreSQL connection

In [5]:
PG_HOST = "postgres"
PG_PORT = 5432
PG_DB = "oilfield"
PG_USER = "analyst"
PG_PASS = "analyst_pass"

engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}")

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT version()"))
    print("PostgreSQL version:", result.fetchone()[0][:50])

PostgreSQL version: PostgreSQL 15.18 (Debian 15.18-1.pgdg13+1) on aarc


In [6]:
# List all tables
query = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public'
ORDER BY table_name
"""
tables = pd.read_sql(text(query), engine)
print(f"Found {len(tables)} tables:")
for t in tables['table_name']:
    print(f"  - {t}")

Found 53 tables:
  - ab_permission
  - ab_permission_view
  - ab_permission_view_role
  - ab_register_user
  - ab_role
  - ab_user
  - ab_user_role
  - ab_view_menu
  - alembic_version
  - callback_request
  - connection
  - dag
  - dag_code
  - dag_owner_attributes
  - dag_pickle
  - dag_run
  - dag_run_note
  - dag_schedule_dataset_reference
  - dag_tag
  - dag_warning
  - dagrun_dataset_event
  - dataset
  - dataset_dag_run_queue
  - dataset_event
  - deliveries
  - drivers
  - import_error
  - job
  - log
  - log_template
  - oil_stations
  - production
  - pump_failures
  - pump_sensors
  - pumps
  - rendered_task_instance_fields
  - serialized_dag
  - session
  - sla_miss
  - slot_pool
  - task_fail
  - task_instance
  - task_instance_note
  - task_map
  - task_outlet_dataset_reference
  - task_reschedule
  - trigger
  - variable
  - vehicles
  - well_targets
  - well_telemetry
  - wells
  - xcom


In [7]:
# Check row counts per table
for table_name in tables['table_name']:
    count = pd.read_sql(text(f"SELECT COUNT(*) as cnt FROM {table_name}"), engine)
    print(f"{table_name}: {count['cnt'].values[0]} rows")

ab_permission: 5 rows
ab_permission_view: 101 rows
ab_permission_view_role: 224 rows
ab_register_user: 0 rows
ab_role: 5 rows
ab_user: 1 rows
ab_user_role: 1 rows
ab_view_menu: 56 rows
alembic_version: 1 rows
callback_request: 0 rows
connection: 59 rows
dag: 3 rows
dag_code: 3 rows
dag_owner_attributes: 0 rows
dag_pickle: 0 rows
dag_run: 0 rows
dag_run_note: 0 rows
dag_schedule_dataset_reference: 0 rows
dag_tag: 6 rows
dagrun_dataset_event: 0 rows
dataset: 0 rows
dataset_dag_run_queue: 0 rows
dataset_event: 0 rows
deliveries: 30 rows
drivers: 5 rows
import_error: 0 rows
job: 1 rows
log: 3 rows
log_template: 2 rows
oil_stations: 20 rows
production: 150 rows
pump_failures: 3 rows
pump_sensors: 72 rows
pumps: 5 rows
rendered_task_instance_fields: 0 rows
serialized_dag: 3 rows
session: 2 rows
sla_miss: 0 rows
slot_pool: 1 rows
task_fail: 0 rows
task_instance: 0 rows
task_instance_note: 0 rows
task_map: 0 rows
task_outlet_dataset_reference: 0 rows
task_reschedule: 0 rows
trigger: 0 rows
var

## 2. MinIO connection

In [8]:
MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
BUCKET_NAME = "oilfield-data"

s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1",
)

print("MinIO client created")

MinIO client created


In [9]:
# List existing buckets
buckets = s3_client.list_buckets()
print("Existing buckets:", [b['Name'] for b in buckets.get('Buckets', [])])

# Create our bucket if not exists
existing = [b['Name'] for b in buckets.get('Buckets', [])]
if BUCKET_NAME not in existing:
    s3_client.create_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket '{BUCKET_NAME}' created")
else:
    print(f"Bucket '{BUCKET_NAME}' already exists")

Existing buckets: []
Bucket 'oilfield-data' created


## 3. Quick data preview from PostgreSQL

In [12]:
# Preview each table
for table_name in tables['table_name']:
    df = pd.read_sql(text(f"SELECT * FROM {table_name} LIMIT 3"), engine)
    print(f"\n=== {table_name} ({len(df.columns)} columns) ===")
    print(df.head(2).to_string())
    print("...")


=== ab_permission (2 columns) ===
   id      name
0   1  can_edit
1   2  can_read
...

=== ab_permission_view (3 columns) ===
   id  permission_id  view_menu_id
0   1              1             4
1   2              2             4
...

=== ab_permission_view_role (3 columns) ===
   id  permission_view_id  role_id
0   1                   1        1
1   2                   2        1
...

=== ab_register_user (8 columns) ===
Empty DataFrame
Columns: [id, first_name, last_name, username, password, email, registration_date, registration_hash]
Index: []
...

=== ab_role (2 columns) ===
   id    name
0   1   Admin
1   2  Public
...

=== ab_user (14 columns) ===
   id first_name last_name username                                                                                                password  active         email                 last_login  login_count  fail_login_count                 created_on                 changed_on created_by_fk changed_by_fk
0   1      Admin      User    adm

## Summary

All connections verified. Ready for ETL and analysis.
- PostgreSQL: connected, all tables populated
- MinIO: connected, bucket created